# Números de Betti — 8 espacios topológicos, 3 métodos

Cálculo de los números de Betti $(b_0, b_1, b_2)$ para:

| # | espacio | $b_0$ | $b_1$ | $b_2$ |
|---|---------|-------|-------|-------|
| 1 | circunferencia $S^1$ | 1 | 1 | 0 |
| 2 | anillo (radio 1.3) | 1 | 1 | 0 |
| 3 | figura 8 | 1 | 2 | 0 |
| 4 | cilindro | 1 | 1 | 0 |
| 5 | toro $T^2$ | 1 | 2 | 1 |
| 6 | banda de Möbius | 1 | 1 | 0 |
| 7 | botella de Klein | 1 | 1 | 0 |
| 8 | plano proyectivo $\mathbb{RP}^2$ | 1 | 0 | 0 |

**Tres métodos, mismos complejos simpliciales:**
- **A mano** (matrices de borde + rangos, `numpy`) → espacios **1, 2, 5, 6**
- Librería **`simplicial`** → espacios **1, 2, 3, 4**
- Librería **`mogutda`** → espacios **5, 6, 7, 8**

> **Nota #6:** "botella de Möbius" no existe como tal — es la **banda / cinta de Möbius** (superficie no orientable con una sola curva frontera). Se trata como tal aquí.

> **Nota clave — cuerpo de coeficientes.** Los números de Betti son rangos de homología **sobre $\mathbb{Q}$**. Los tres métodos (y ambas librerías) calculan sobre $\mathbb{Q}$, así que la **torsión no se ve**: $H_1(\text{Klein})=\mathbb{Z}\oplus\mathbb{Z}/2$ y $H_1(\mathbb{RP}^2)=\mathbb{Z}/2$, pero sobre $\mathbb{Q}$ dan $b_1=1$ y $b_1=0$. Sobre $\mathbb{F}_2$ los números cambiarían (Klein $\to (1,2,1)$, $\mathbb{RP}^2 \to (1,1,1)$). Ver la última sección.


## 0. Instalación (Colab)

In [ ]:
!pip install simplicial mogutda numpy -q

## 1. Triangulaciones (compartidas por los tres métodos)

Cada espacio se define por sus **símplices maximales** (los triángulos de más alta dimensión, o aristas si es un grafo). Todas las caras se derivan automáticamente. Usar las **mismas** triangulaciones en los tres métodos hace la comparación honesta.

- Circunferencia / figura 8: grafos (solo aristas).
- Anillo, cilindro, Möbius: superficies con frontera.
- Toro y Klein: malla $3\times3$ (mínimo para que la identificación sea un complejo simplicial válido). El toro identifica ambas direcciones "rectas"; Klein invierte una (`j -> n-1-j`) al dar la vuelta.
- $\mathbb{RP}^2$: triangulación mínima de 6 vértices (cociente antipodal del icosaedro / hemi-icosaedro).


In [ ]:
from itertools import combinations

def torus_grid(n=3):
    """Malla n x n identificada como toro. vértice(i,j) = (i%n)*n + (j%n)."""
    V = lambda i, j: (i % n) * n + (j % n)
    T = []
    for i in range(n):
        for j in range(n):
            a, b, c, d = V(i, j), V(i+1, j), V(i, j+1), V(i+1, j+1)
            T += [tuple(sorted((a, b, d))), tuple(sorted((a, d, c)))]
    return sorted({t for t in T if len(set(t)) == 3})

def klein_grid(n=3):
    """Malla n x n identificada como botella de Klein (una dirección invertida)."""
    def W(i, j):
        if i >= n:          # al pasar el borde superior, se invierte j
            i %= n; j = n - 1 - j
        return (i % n) * n + (j % n)
    T = []
    for i in range(n):
        for j in range(n):
            a, b, c, d = W(i, j), W(i+1, j), W(i, j+1), W(i+1, j+1)
            T += [tuple(sorted((a, b, d))), tuple(sorted((a, d, c)))]
    return sorted({t for t in T if len(set(t)) == 3})

SPACES = {
    "1. circunferencia S^1": [(0,1), (1,2), (0,2)],
    "2. anillo (r=1.3)":     [(0,1,3), (1,4,3), (1,2,4), (2,5,4), (2,0,5), (0,3,5)],
    "3. figura 8":           [(0,1), (1,2), (0,2), (0,3), (3,4), (0,4)],
    "4. cilindro":           [(0,1,3), (1,3,4), (1,2,4), (2,4,5), (2,0,5), (0,5,3)],
    "5. toro T^2":           torus_grid(3),
    "6. banda de Möbius":    [(0,1,2), (1,2,3), (2,3,4), (0,3,4), (0,1,4)],
    "7. botella de Klein":   klein_grid(3),
    "8. plano proy. RP^2":   [(0,1,2),(0,1,3),(0,2,4),(0,3,5),(0,4,5),
                              (1,2,5),(1,3,4),(1,4,5),(2,3,4),(2,3,5)],
}

for name, simp in SPACES.items():
    print(f"{name:24}  {len(simp)} símplices maximales")


## Método A — A mano (matrices de borde) → espacios 1, 2, 5, 6

**Idea.** Sobre $\mathbb{Q}$, el $k$-ésimo número de Betti es el rango de $H_k = \ker\partial_k / \operatorname{im}\partial_{k+1}$:

$$b_k = \dim\ker\partial_k - \operatorname{rank}\partial_{k+1} = \big(n_k - \operatorname{rank}\partial_k\big) - \operatorname{rank}\partial_{k+1}$$

donde $n_k$ = número de $k$-símplices y $\partial_k : C_k \to C_{k-1}$ es la matriz de borde. Por convención $\partial_0 = 0$, así que $b_0 = n_0 - \operatorname{rank}\partial_1$ = número de componentes conexas.

La matriz de borde de un $k$-símplice $[v_0,\dots,v_k]$ tiene entradas $(-1)^i$ en cada cara $[v_0,\dots,\hat{v_i},\dots,v_k]$.


In [ ]:
import numpy as np

def simplicial_closure(maximal):
    """Devuelve {dim: [símplices ordenados]} incluyendo TODAS las caras."""
    allsimp = set()
    for s in maximal:
        s = tuple(sorted(s))
        for k in range(1, len(s) + 1):
            for f in combinations(s, k):
                allsimp.add(f)
    by_dim = {}
    for s in allsimp:
        by_dim.setdefault(len(s) - 1, []).append(s)
    return {k: sorted(v) for k, v in by_dim.items()}

def boundary_matrix(by_dim, k):
    """Matriz de borde d_k : C_k -> C_{k-1} sobre Q."""
    if k == 0 or k not in by_dim:
        return np.zeros((0, 0))
    ksimp = by_dim[k]
    faces = by_dim.get(k - 1, [])
    idx = {s: i for i, s in enumerate(faces)}
    M = np.zeros((len(faces), len(ksimp)))
    for col, s in enumerate(ksimp):
        for i in range(len(s)):
            face = s[:i] + s[i+1:]
            M[idx[face], col] = (-1) ** i
    return M

def betti_by_hand(maximal, maxdim=2, verbose=False):
    by_dim = simplicial_closure(maximal)
    n    = {k: len(by_dim.get(k, [])) for k in range(maxdim + 2)}
    rank = {}
    for k in range(maxdim + 2):
        M = boundary_matrix(by_dim, k)
        rank[k] = int(np.linalg.matrix_rank(M)) if M.size else 0
    betti = {k: n.get(k, 0) - rank.get(k, 0) - rank.get(k + 1, 0) for k in range(maxdim + 1)}
    if verbose:
        for k in range(maxdim + 1):
            print(f"  dim {k}: n_{k}={n.get(k,0):2d}  rank(∂_{k})={rank.get(k,0):2d}  "
                  f"rank(∂_{k+1})={rank.get(k+1,0):2d}  ->  b_{k} = {betti[k]}")
    return tuple(betti[k] for k in range(maxdim + 1))

# --- demostración detallada con la circunferencia ---
print("Circunferencia S^1 (triángulo hueco), paso a paso:")
by_dim = simplicial_closure(SPACES["1. circunferencia S^1"])
print("  ∂_1 (filas=vértices, columnas=aristas):")
print(boundary_matrix(by_dim, 1).astype(int))
betti_by_hand(SPACES["1. circunferencia S^1"], verbose=True)

# --- aplicar a los espacios 1, 2, 5, 6 ---
por_mano = ["1. circunferencia S^1", "2. anillo (r=1.3)", "5. toro T^2", "6. banda de Möbius"]
print("\n=== Método A (a mano) ===")
resultados_mano = {}
for name in por_mano:
    b = betti_by_hand(SPACES[name])
    resultados_mano[name] = b
    print(f"{name:24}  (b0,b1,b2) = {b}")


## Método B — librería `simplicial` → espacios 1, 2, 3, 4

`simplicial` (Simon Dobson). Se añaden los vértices con `addSimplex(id=...)` y luego los símplices maximales con `addSimplexWithBasis([...])`, que **crea automáticamente todas las caras**. `bettiNumbers()` devuelve un dict `{dim: b_k}` (sobre $\mathbb{Q}$).


In [ ]:
from simplicial import SimplicialComplex as SimplicialSC

def betti_simplicial(maximal, maxdim=2):
    c = SimplicialSC()
    vertices = {v for s in maximal for v in s}
    for v in vertices:
        c.addSimplex(id=int(v))
    for s in maximal:
        c.addSimplexWithBasis([int(x) for x in s])   # genera aristas/caras solo
    b = c.bettiNumbers()
    return tuple(int(b.get(k, 0)) for k in range(maxdim + 1))

primeros_cuatro = ["1. circunferencia S^1", "2. anillo (r=1.3)", "3. figura 8", "4. cilindro"]
print("=== Método B (librería simplicial) ===")
resultados_simplicial = {}
for name in primeros_cuatro:
    b = betti_simplicial(SPACES[name])
    resultados_simplicial[name] = b
    print(f"{name:24}  (b0,b1,b2) = {b}")


## Método C — librería `mogutda` → espacios 5, 6, 7, 8

`mogutda`. El constructor `SimplicialComplex(simplices=[...])` recibe los símplices maximales como tuplas y **deriva todas las caras**. `betti_number(k)` devuelve $b_k$ (sobre $\mathbb{Q}$).


In [ ]:
from mogutda import SimplicialComplex as MogutdaSC

def betti_mogutda(maximal, maxdim=2):
    sc = MogutdaSC(simplices=[tuple(int(x) for x in s) for s in maximal])
    return tuple(int(sc.betti_number(k)) for k in range(maxdim + 1))

ultimos_cuatro = ["5. toro T^2", "6. banda de Möbius", "7. botella de Klein", "8. plano proy. RP^2"]
print("=== Método C (librería mogutda) ===")
resultados_mogutda = {}
for name in ultimos_cuatro:
    b = betti_mogutda(SPACES[name])
    resultados_mogutda[name] = b
    print(f"{name:24}  (b0,b1,b2) = {b}")


## Resumen — tabla comparativa

In [ ]:
esperado = {
    "1. circunferencia S^1": (1,1,0), "2. anillo (r=1.3)": (1,1,0),
    "3. figura 8": (1,2,0),           "4. cilindro": (1,1,0),
    "5. toro T^2": (1,2,1),           "6. banda de Möbius": (1,1,0),
    "7. botella de Klein": (1,1,0),   "8. plano proy. RP^2": (1,0,0),
}
mano       = {**resultados_mano}
simplicial = {**resultados_simplicial}
mogutda    = {**resultados_mogutda}

print(f"{'espacio':24} {'esperado':11} {'a mano':11} {'simplicial':11} {'mogutda':11}")
print("-"*72)
for name in SPACES:
    e = esperado[name]
    m  = mano.get(name, "—")
    sd = simplicial.get(name, "—")
    mg = mogutda.get(name, "—")
    print(f"{name:24} {str(e):11} {str(m):11} {str(sd):11} {str(mg):11}")


## Apéndice — por qué Klein y $\mathbb{RP}^2$ "esconden" información

Los números de Betti de arriba son sobre $\mathbb{Q}$. La homología entera completa es:

| espacio | $H_0$ | $H_1$ | $H_2$ |
|---------|-------|-------|-------|
| toro | $\mathbb{Z}$ | $\mathbb{Z}^2$ | $\mathbb{Z}$ |
| Klein | $\mathbb{Z}$ | $\mathbb{Z}\oplus\mathbb{Z}/2$ | $0$ |
| $\mathbb{RP}^2$ | $\mathbb{Z}$ | $\mathbb{Z}/2$ | $0$ |

El $\mathbb{Z}/2$ es **torsión**: desaparece al tensar con $\mathbb{Q}$ (de ahí $b_1=1$ para Klein y $b_1=0$ para $\mathbb{RP}^2$), pero **sobrevive** sobre $\mathbb{F}_2$. Cambiando el cuerpo a $\mathbb{F}_2$ los rangos serían Klein $=(1,2,1)$ y $\mathbb{RP}^2=(1,1,1)$.

La celda siguiente re-hace el cálculo "a mano" sobre $\mathbb{F}_2$ (eliminación gaussiana módulo 2) para mostrar la diferencia. Confirma que la torsión que no ven los métodos A/B/C está realmente ahí.


In [ ]:
def rank_mod2(M):
    """Rango de una matriz 0/1 sobre F_2 por eliminación gaussiana."""
    A = (np.array(M) % 2).astype(int).copy()
    if A.size == 0:
        return 0
    rows, cols = A.shape
    r = 0
    for c in range(cols):
        piv = None
        for i in range(r, rows):
            if A[i, c] % 2 == 1:
                piv = i; break
        if piv is None:
            continue
        A[[r, piv]] = A[[piv, r]]
        for i in range(rows):
            if i != r and A[i, c] % 2 == 1:
                A[i] = (A[i] + A[r]) % 2
        r += 1
    return r

def betti_F2(maximal, maxdim=2):
    by_dim = simplicial_closure(maximal)
    n    = {k: len(by_dim.get(k, [])) for k in range(maxdim + 2)}
    rank = {k: rank_mod2(boundary_matrix(by_dim, k)) for k in range(maxdim + 2)}
    return tuple(n.get(k,0) - rank.get(k,0) - rank.get(k+1,0) for k in range(maxdim + 1))

print(f"{'espacio':24} {'sobre Q':11} {'sobre F2':11}")
print("-"*48)
for name in ["5. toro T^2", "7. botella de Klein", "8. plano proy. RP^2"]:
    print(f"{name:24} {str(betti_by_hand(SPACES[name])):11} {str(betti_F2(SPACES[name])):11}")
